# 15. Samplers and solvers — DPM-Solver++ and UniPC with a real diffusion model path

The old `toy_data_prediction` oracle is removed. A small trainable data-prediction network is first fitted for five CPU steps on a VP diffusion path, and the same model is then used by DPM-Solver++ and UniPC.

Only data dimension, hidden width, batch size, and solver history length are reduced. The solver formulas are not replaced by generic Euler/Heun updates.


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(5)
device = torch.device("cpu")
print("device:", device)


## 1. VP schedule and log-SNR coordinate


In [ ]:
def alpha(t):
    return torch.cos(0.5 * math.pi * t)


def sigma(t):
    return torch.sin(0.5 * math.pi * t)


def lambda_t(t):
    return torch.log(alpha(t)) - torch.log(sigma(t))


def inverse_lambda(value):
    return 2.0 / math.pi * torch.atan(torch.exp(-value))


## 2. Small data-prediction diffusion network

DPM-Solver++ is naturally written in the data-prediction parameterization. This network predicts `x0` from `(x_t,t)`; it is a genuine learned model path rather than an analytic placeholder.


In [ ]:
class TinyDataPredictor(nn.Module):
    def __init__(self, data_dim=2, hidden_dim=24):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(data_dim + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, data_dim),
        )

    def forward(self, x_t, t):
        model_input = torch.cat([x_t, t[:, None]], dim=-1)
        return self.net(model_input)


model = TinyDataPredictor().to(device)
clean = torch.randn(16, 2, device=device)
noise = torch.randn_like(clean)
train_t = torch.linspace(0.05, 0.95, 16, device=device)

noisy = (
    alpha(train_t)[:, None] * clean
    + sigma(train_t)[:, None] * noise
)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
loss_history = []

for step in range(5):
    optimizer.zero_grad()
    prediction = model(noisy, train_t)
    loss = F.mse_loss(prediction, clean)
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())
    print(f"step {step + 1}: data-prediction loss={loss.item():.6f}")

print("loss history:", loss_history)


## 3. DPM-Solver++ first- and second-order updates

The update is performed in log-SNR space and uses model data predictions directly. The midpoint evaluation is retained for the second-order solver.


In [ ]:
def model_data_prediction(x, t):
    if t.ndim == 0:
        t = t.expand(x.size(0))
    return model(x, t)


def dpmpp_first_order(x_s, s, t, model_fn):
    h = lambda_t(t) - lambda_t(s)
    model_s = model_fn(x_s, s)
    phi_1 = torch.expm1(-h)

    return (
        sigma(t) / sigma(s) * x_s
        - alpha(t) * phi_1 * model_s
    )


def dpmpp_second_order(x_s, s, t, model_fn, r1=0.5):
    lambda_s = lambda_t(s)
    h = lambda_t(t) - lambda_s
    s1 = inverse_lambda(lambda_s + r1 * h)

    model_s = model_fn(x_s, s)
    x_s1 = (
        sigma(s1) / sigma(s) * x_s
        - alpha(s1) * torch.expm1(-r1 * h) * model_s
    )
    model_s1 = model_fn(x_s1, s1)

    phi_1 = torch.expm1(-h)
    correction = (
        0.5
        / r1
        * alpha(t)
        * phi_1
        * (model_s1 - model_s)
    )

    return (
        sigma(t) / sigma(s) * x_s
        - alpha(t) * phi_1 * model_s
        - correction
    )


x_s = torch.randn(2, 2, device=device)
s = torch.tensor(0.80, device=device)
t = torch.tensor(0.60, device=device)

print("DPM++ first order:", dpmpp_first_order(x_s, s, t, model_data_prediction))
print("DPM++ second order:", dpmpp_second_order(x_s, s, t, model_data_prediction))


## 4. UniPC B(h) multistep predictor

This keeps the actual UniP control structure used in the reference implementations: previous model outputs in log-SNR coordinates, normalized history positions `r_k`, finite differences `D1`, the `R` system, phi-function recursion, and the B(h) coefficient. Order 2 uses the reference simplification `rho=1/2`; higher orders solve the coefficient system.


In [ ]:
def unip_bh_predict(
    x,
    current_time,
    target_time,
    model_history,
    time_history,
    order,
):
    if order < 1:
        raise ValueError("order must be positive")
    if order > len(model_history):
        raise ValueError("order exceeds available model history")

    model_s0 = model_history[-1]
    lambda_s0 = lambda_t(current_time)
    lambda_target = lambda_t(target_time)
    h = lambda_target - lambda_s0

    rks = []
    d1_terms = []
    for history_offset in range(1, order):
        previous_time = time_history[-(history_offset + 1)]
        previous_model = model_history[-(history_offset + 1)]
        lambda_previous = lambda_t(previous_time)

        rk = (lambda_previous - lambda_s0) / h
        rks.append(rk)
        d1_terms.append((previous_model - model_s0) / rk)

    rks.append(torch.ones((), device=x.device))
    rks = torch.stack(rks)

    hh = -h
    h_phi_1 = torch.expm1(hh)
    h_phi_k = h_phi_1 / hh - 1.0
    b_h = torch.expm1(hh)

    factorial = 1.0
    matrix_rows = []
    rhs = []
    for power in range(1, order + 1):
        matrix_rows.append(rks.pow(power - 1))
        rhs.append(h_phi_k * factorial / b_h)

        factorial *= power + 1
        h_phi_k = h_phi_k / hh - 1.0 / factorial

    coefficient_matrix = torch.stack(matrix_rows)
    rhs = torch.stack(rhs)

    if d1_terms:
        differences = torch.stack(d1_terms, dim=0)

        if order == 2:
            rho = torch.full(
                (1,),
                0.5,
                dtype=x.dtype,
                device=x.device,
            )
        else:
            rho = torch.linalg.solve(
                coefficient_matrix[:-1, :-1],
                rhs[:-1],
            ).to(x.dtype)

        predictor_residual = torch.einsum(
            "k,kbd->bd",
            rho,
            differences,
        )
    else:
        predictor_residual = torch.zeros_like(x)

    base = (
        sigma(target_time) / sigma(current_time) * x
        - alpha(target_time) * h_phi_1 * model_s0
    )
    return base - alpha(target_time) * b_h * predictor_residual


## 5. UniP history reuse with the trained model

The first step starts at order 1; subsequent steps use as much previous model history as is available, up to order 3. This is a solver execution check, not an additional training loop.


In [ ]:
sample = torch.randn(2, 2, device=device)
times = [
    torch.tensor(value, device=device)
    for value in [0.90, 0.75, 0.60, 0.45]
]

model_history = []
time_history = []

for step_index in range(len(times) - 1):
    current_time = times[step_index]
    target_time = times[step_index + 1]

    current_model = model_data_prediction(sample, current_time).detach()
    model_history.append(current_model)
    time_history.append(current_time)

    order = min(3, len(model_history))
    sample = unip_bh_predict(
        sample,
        current_time,
        target_time,
        model_history,
        time_history,
        order=order,
    )

    print(
        f"UniP step {step_index + 1}: order={order}, "
        f"sample_norm={sample.norm().item():.6f}"
    )


## References and provenance

- DPM-Solver++: data-prediction formulation and log-SNR analytical updates.
- UniPC reference implementation / Diffusers UniPCMultistepScheduler: normalized log-SNR history, `D1` differences, B(h), phi recursion, and the linear system for arbitrary multistep order.

The old analytic toy oracle is completely removed. Only tensor scale and training budget are reduced.
